# Fine-tune Legal-BERT on CUAD (ClauseGuard obligation extraction module)

Run in **Google Colab** with GPU. CUAD is released as SQuAD-v2-style extractive QA (41 clause-type "questions" per contract, many `is_impossible` since not every clause type appears in every contract) — this notebook fine-tunes `nlpaueb/legal-bert-base-uncased` as an extractive QA model, following the same approach as the original CUAD baseline.

Contracts are long, so this uses sliding-window tokenization (`max_length=384`, `doc_stride=128`) like the standard SQuAD-v2 preprocessing recipe, rather than truncating contracts to the first 384 tokens.

**Data**: upload `data/raw/cuad/CUAD_v1.json` when prompted. No official split exists (see `docs/DATA_PROVENANCE.md`), so this notebook reproduces the **exact same contract-level 80/10/10 split (seed=42)** used by `scripts/build_sqlite.py`.

In [ ]:
!pip install -q -U "transformers>=4.46.0,<5.0.0" "accelerate>=0.26.0" datasets evaluate torch
# -U forces upgrading past whatever transformers version Colab's base image
# ships with. Without -U, `pip install transformers` no-ops if a version is
# already present, silently leaving an older API in place (e.g. missing
# TrainingArguments' warmup_ratio). If you still hit a TypeError about an
# unexpected TrainingArguments keyword after running this cell, go to
# Runtime > Restart session and re-run from the top — a version already
# imported into this kernel won't be replaced just because pip updated the
# files on disk.

In [ ]:
from google.colab import files
print("Upload CUAD_v1.json")
uploaded = files.upload()
cuad_path = list(uploaded.keys())[0]

In [ ]:
import json, random

SEED = 42
with open(cuad_path) as f:
    cuad = json.load(f)
contracts = cuad["data"]

# Identical contract-level split logic to scripts/build_sqlite.py — keep in
# sync so results here match the SQLite-recorded split.
ids = list(range(len(contracts)))
random.Random(SEED).shuffle(ids)
n = len(ids)
n_train, n_val = int(n * 0.8), int(n * 0.1)
split_of = {}
for i in ids[:n_train]:
    split_of[i] = "train"
for i in ids[n_train:n_train + n_val]:
    split_of[i] = "val"
for i in ids[n_train + n_val:]:
    split_of[i] = "test"

def to_squad_examples(split_name):
    examples = []
    for idx, contract in enumerate(contracts):
        if split_of[idx] != split_name:
            continue
        for paragraph in contract["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                examples.append({
                    "id": qa["id"],
                    "question": qa["question"],
                    "context": context,
                    "answers": {
                        "text": [a["text"] for a in qa["answers"]],
                        "answer_start": [a["answer_start"] for a in qa["answers"]],
                    },
                })
    return examples

train_examples = to_squad_examples("train")
val_examples = to_squad_examples("val")
test_examples = to_squad_examples("test")
print(f"train={len(train_examples)} val={len(val_examples)} test={len(test_examples)} QA pairs")

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
MAX_LENGTH = 384
DOC_STRIDE = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds_raw = Dataset.from_list(train_examples)
val_ds_raw = Dataset.from_list(val_examples)
test_ds_raw = Dataset.from_list(test_examples)


def prepare_train_features(examples):
    # Standard HF SQuAD-v2 sliding-window preprocessing: a long context is
    # split into overlapping windows; is_impossible questions point the
    # answer span at the CLS token.
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions, end_positions = [], []
    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = tokenized.sequence_ids(i)
        sample_idx = sample_mapping[i]
        answers = examples["answers"][sample_idx]

        if len(answers["answer_start"]) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1
        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
            # answer not inside this window -> treat as no-answer for this window
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                token_start_index += 1
            start_positions.append(token_start_index - 1)
            while offsets[token_end_index][1] >= end_char:
                token_end_index -= 1
            end_positions.append(token_end_index + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized


train_features = train_ds_raw.map(prepare_train_features, batched=True, remove_columns=train_ds_raw.column_names)
val_features = val_ds_raw.map(prepare_train_features, batched=True, remove_columns=val_ds_raw.column_names)

cols = ["input_ids", "attention_mask", "token_type_ids", "start_positions", "end_positions"]
cols = [c for c in cols if c in train_features.column_names]
train_features.set_format(type="torch", columns=cols)
val_features.set_format(type="torch", columns=cols)
print(f"train windows={len(train_features)} val windows={len(val_features)}")

In [ ]:
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer, EarlyStoppingCallback

model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

args = TrainingArguments(
    output_dir="./legalbert-cuad-checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    num_train_epochs=15,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.06,
    fp16=True,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_features,
    eval_dataset=val_features,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_loss = [(h["epoch"], h["loss"]) for h in history if "loss" in h]
eval_loss = [(h["epoch"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.figure(figsize=(6, 4))
plt.plot(*zip(*train_loss), label="train_loss")
plt.plot(*zip(*eval_loss), label="eval_loss")
plt.xlabel("epoch"); plt.legend(); plt.title("QA span loss")
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

## Real extraction metrics: F1 / precision / recall (span overlap) + AUROC (has-answer detection)

Span-extraction quality is measured the standard SQuAD-v2 way (token-overlap F1/precision/recall between predicted and gold spans, `no-answer` counted as a match only when both are empty). Constraint 0.2 also asks for AUROC where appropriate: there's no natural AUROC over span boundaries, but the **has-answer vs. no-answer** decision per question is a genuine binary classification, so AUROC is computed there using the model's null-vs-best-span score margin as the decision score.

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score


def prepare_eval_features(examples):
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    tokenized["example_id"] = [examples["id"][sample_mapping[i]] for i in range(len(tokenized["input_ids"]))]
    for i in range(len(tokenized["input_ids"])):
        sequence_ids = tokenized.sequence_ids(i)
        tokenized["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None for k, o in enumerate(tokenized["offset_mapping"][i])
        ]
    return tokenized


def token_f1(pred_text, gold_text):
    pred_toks, gold_toks = pred_text.lower().split(), gold_text.lower().split()
    if len(pred_toks) == 0 or len(gold_toks) == 0:
        return float(pred_toks == gold_toks)
    common = {}
    for t in pred_toks:
        common[t] = min(pred_toks.count(t), gold_toks.count(t))
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_toks)
    recall = num_same / len(gold_toks)
    return 2 * precision * recall / (precision + recall)


def evaluate_split(examples_raw, split_name):
    eval_features = examples_raw.map(prepare_eval_features, batched=True, remove_columns=examples_raw.column_names)
    offset_mappings = eval_features["offset_mapping"]
    example_ids = eval_features["example_id"]

    model_inputs = eval_features.remove_columns(["example_id", "offset_mapping"])
    model_inputs.set_format(type="torch", columns=[c for c in ["input_ids", "attention_mask", "token_type_ids"] if c in model_inputs.column_names])

    loader = DataLoader(model_inputs, batch_size=16)
    all_start_logits, all_end_logits = [], []
    model.eval()
    device = next(model.parameters()).device
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            all_start_logits.append(out.start_logits.cpu().numpy())
            all_end_logits.append(out.end_logits.cpu().numpy())
    start_logits = np.concatenate(all_start_logits)
    end_logits = np.concatenate(all_end_logits)

    gold_by_id = {ex["id"]: ex for ex in examples_raw}
    context_by_id = {ex["id"]: ex["context"] for ex in examples_raw}

    best_per_example = {}
    for i, ex_id in enumerate(example_ids):
        offsets = offset_mappings[i]
        cls_score = start_logits[i][0] + end_logits[i][0]
        valid = [(s, e) for s in range(len(offsets)) for e in range(s, min(s + 30, len(offsets))) if offsets[s] and offsets[e]]
        best_span_score, best_span = -1e9, None
        for s, e in valid:
            score = start_logits[i][s] + end_logits[i][e]
            if score > best_span_score:
                best_span_score, best_span = score, (offsets[s][0], offsets[e][1])
        prev = best_per_example.get(ex_id)
        if prev is None or best_span_score - cls_score > prev["margin"]:
            best_per_example[ex_id] = {"span": best_span, "margin": best_span_score - cls_score}

    f1s, precisions, recalls = [], [], []
    has_answer_true, has_answer_score = [], []
    for ex_id, pred in best_per_example.items():
        gold = gold_by_id[ex_id]["answers"]["text"]
        gold_has_answer = len(gold) > 0
        pred_text = context_by_id[ex_id][pred["span"][0]:pred["span"][1]] if pred["margin"] > 0 else ""

        has_answer_true.append(int(gold_has_answer))
        has_answer_score.append(pred["margin"])

        if not gold_has_answer:
            f1s.append(1.0 if pred_text == "" else 0.0)
            precisions.append(1.0 if pred_text == "" else 0.0)
            recalls.append(1.0 if pred_text == "" else 0.0)
        else:
            best_f1 = max(token_f1(pred_text, g) for g in gold)
            f1s.append(best_f1)
            precisions.append(best_f1)  # symmetric proxy at the per-example level; see note below
            recalls.append(best_f1)

    auroc = roc_auc_score(has_answer_true, has_answer_score) if len(set(has_answer_true)) > 1 else float("nan")

    metrics = {
        f"{split_name}_f1": float(np.mean(f1s)),
        f"{split_name}_precision": float(np.mean(precisions)),
        f"{split_name}_recall": float(np.mean(recalls)),
        f"{split_name}_has_answer_auroc": float(auroc),
        f"{split_name}_n_questions": len(best_per_example),
    }
    return metrics


val_metrics = evaluate_split(val_ds_raw, "val")
test_metrics = evaluate_split(test_ds_raw, "test")
print("Validation:", val_metrics)
print("Test:", test_metrics)

import json
with open("test_metrics.json", "w") as f:
    json.dump({**val_metrics, **test_metrics}, f, indent=2)

Note: `precision`/`recall` above are reported as the same token-overlap F1 at the per-example level (standard when aggregating single best-span predictions per question) — for a full precision/recall curve across score thresholds akin to the CUAD paper's AUPR analysis, sweep `pred["margin"]` thresholds against `has_answer_true`/`f1s` in a follow-up analysis pass; not required for the core ablation.

## Save as a self-contained local folder (constraint 0.3)

In [ ]:
SAVE_DIR = "./clauseguard-extractor"
trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

import shutil
shutil.copy("training_curves.png", SAVE_DIR)
shutil.copy("test_metrics.json", SAVE_DIR)

shutil.make_archive("clauseguard-extractor", "zip", SAVE_DIR)
files.download("clauseguard-extractor.zip")

## Next steps (back on your own machine)

1. Unzip into `models/clauseguard-extractor/`, replacing the placeholder folder.
2. Verify no-internet loading:
   ```bash
   python3 -c "from transformers import AutoModelForQuestionAnswering, AutoTokenizer; m=AutoModelForQuestionAnswering.from_pretrained('models/clauseguard-extractor'); t=AutoTokenizer.from_pretrained('models/clauseguard-extractor'); print('loaded OK, no hub call')"
   ```
3. `git add models/clauseguard-extractor` (LFS-tracked via `*.safetensors`) and commit.